In [1]:
import random
import json
from pathlib import Path
from enum import Enum
from typing import Tuple, List, Optional, Callable, Union, Literal

import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2

In [3]:
class SplitType(Enum):
    TRAIN = 'train'
    VAL = 'val'
    TEST ='test'

class Sentinel(Dataset):
    def __init__(self, root_dir:Union[str, Path], split_type: Optional[str]=None, transform: Optional[Callable]= None, split_mode: Literal['random', 'split']='random', split_ratio: Tuple[float, float, float]=(0.7, 0.15, 0.15), split_file: Optional[Union[str, Path]]=None, seed: int=42):
        self.root_dir =Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset root dir not found {self.root_dir}")
        
        #convert string split_type to enum
        self.split_type = SplitType(split_type) if split_type else None

        #default transform pipeline
        self.transform = transform if transform else v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True)
        ])

        #collecting image pairs
        self.all_image_pairs = self._collect_images()

        #apply split if specified
        if split_type:
            if split_mode =='split' and split_file:
                self.image_pairs =self._apply_predefined_split(split_file)
            elif split_mode == 'random':
                self.image_pairs = self._apply_random_split(split_ratio, seed)
            else:
                raise ValueError("Invalid split config")
        else:
            # if no split type specified use all images
            self.image_pairs = self.all_image_pairs
        
        print(f'Total image pairs found: {len(self)}')

    def _collect_images(self) -> List[Tuple[Path, Path]]:
        #collected pairs s1 and optical s2 image path from dir
        image_pairs = []

        for category in self.root_dir.iterdir():
            if not category.is_dir():
                continue

            s1_path = category / 's1'
            s2_path =category / 's2'

            if not (s1_path.is_dir() and s2_path.is_dir()):
                continue

            #collect pairs
            for s1_file in s1_path.glob('*.png'):
                s2_filename = list(s1_file.name.split('_'))
                s2_filename[2] = 's2'
                s2_file =s2_path / '_'.join(s2_filename)

                if not s2_file.exists():
                    continue

                image_pairs.append((s1_file, s2_file))
            return image_pairs
        
        def predefined_split(self, split_file:Union[str, Path])-> List[Tuple[Path, Path]]:
            try:
                with open(split_file,'r') as f:
                    splits = json.load(f)
                
                if self.split_type.value not in splits['data']:
                    raise ValueError(f"Split type {self.split_type.value} not found in split file")
            
                split_filenames = set(splits['data'][self.split_type.value]) # data['split']
                return [pair for pair in self.all_image_pairs 
                    if any(p.name in split_filenames for p in pair[:2])]
            
            except Exception as e:
                print(f'Could not open split file\n\t{e}')
                raise
        
        def apply_random_split(self, split_ratio: Tuple[float, float, float], seed: int)-> List[Tuple[Path, Path]]:
            if sum(split_ratio) != 1:
                raise ValueError("Split ratios must sum to 1")
        
            # set random seed for reproducibility
            random.seed(seed)
        
            # shuffle indices
            indices = list(range(len(self.all_image_pairs)))
            random.shuffle(indices)
        
            # calculate split points
            train_end = int(len(indices) * split_ratio[0])
            val_end = train_end + int(len(indices) * split_ratio[1])
        
            # select appropriate slice based on split type
            if self.split_type == SplitType.TRAIN:
                split_indices = indices[:train_end]
            elif self.split_type == SplitType.VAL:
                split_indices = indices[train_end:val_end]
            else:  # TEST
                split_indices = indices[val_end:]
            
            return [self.all_image_pairs[i] for i in split_indices]
        
        def save_split(self, output_file: Union[str, Path], append:bool =False):
            if self.split_type:
                split = self.split_type.value
                split_info = {
                    'data': {
                        split: [P[0].name for p in self.image_pairs]
                    }
                }

                mode= 'a' if append else 'w'
                with open(output_file, mode) as f:
                    json.dump(split_info, f, indent=2)
        
        def __len__(self):
            return len(self.image_pairs)
        
        def __getitem__(self, idx:int)-> Tuple[torch.Tensor, torch.Tensor]:
            s1_path, s2_path =self.image_pairs[idx]

            #load images
            s1_image = Image.open(s1_path).convert('RGB')
            s2_image = Image.open(s2_path).convert('RGB')

            #applly transform
            s1_image = self.transform(s1_image)
            s2_image = self.transform(s2_image)
        
            return s1_image, s2_image
            
            
            

## PIX2PIX Model implementation

In [4]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Downsample(nn.Module): #it consists of convolution batchnorm relu layer with k filter
    def __init__(self, c_in, c_out, kernel_size=4, stride=2, padding=1, negative_slope=0.02, use_norm=True):
        #intialzies the unet downsample block
        # c_in (int): The number of input channels.
        # c_out (int): The number of output channels.
        # kernel_size (int, optional): The size of the convolving kernel. Default is 4.
        # stride (int, optional): Stride of the convolution. Default is 2.
        # padding (int, optional): Zero-padding added to both sides of the input. Default is 0.
        # negative_slope (float, optional): Negative slope for the LeakyReLU activation function. Default is 0.2.
        # use_norm (bool, optinal): If use norm layer. If True add a BatchNorm layer after Conv. Default is True.

        super(Downsample, self).__init__()
        block =[]
        block += [nn.Convo2d(in_channel=c_in, out_channel=c_out, kernel_size=kernel_size, sttride= stride, padding=padding, bias=(not use_norm))]

        if use_norm:
            block += [nn.BatchNoemd(num_features=c_out)]
        
        block += [nn.LeakyReLU(negative_slope = negative_slope)]

        self.conv_block = nn.Sequential(*block)

    
    def forward(self,x) :
        return self.conv_block(x)
    